In [0]:
CREATE OR REPLACE VIEW meridian_t12.gold.booking_metrics_view 
COMMENT 'Metric view for analyzing booking performance and revenue KPIs.'
AS SELECT 
    COALESCE(p.property_type, 'Unknown') AS property_type,
    COALESCE(u.country, 'Unknown') AS country,
    COALESCE(d.destination, 'Unknown') AS destination,
    COUNT(f.booking_id) AS total_bookings,
    SUM(f.amount) AS total_revenue,
    AVG(f.amount) AS average_booking_value,
    COUNT(DISTINCT f.user_id) AS unique_customers,
    (SUM(f.amount) / COUNT(f.booking_id)) AS revenue_per_booking_ratio
FROM meridian_t12.gold.gold_fact_bookings f
LEFT JOIN (
    -- Deduplicate SCD2 properties to get 1 row per property_id
    SELECT DISTINCT property_id, property_type 
    FROM meridian_t12.gold.gold_dim_properties
) p ON f.property_id = p.property_id
LEFT JOIN meridian_t12.gold.gold_dim_users u ON f.user_id = u.user_id
LEFT JOIN meridian_t12.gold.gold_dim_destinations d ON f.destination_id = d.destination_id
GROUP BY 
    COALESCE(p.property_type, 'Unknown'),
    COALESCE(u.country, 'Unknown'),
    COALESCE(d.destination, 'Unknown');